### Fact Tables

In [0]:
order_silver = spark.table("02_silver_catalog.silver.order")
invoice_silver = spark.table("02_silver_catalog.silver.invoice")
estimate_silver = spark.table("02_silver_catalog.silver.estimate")
customer_survey_silver = spark.table("02_silver_catalog.silver.customer_survey")
store_silver = spark.table("02_silver_catalog.silver.store")
ns_budget_silver = spark.table("02_silver_catalog.silver.ns_budget")

In [0]:
gold_catalog = "03_gold_catalog.dim_fact_tables."

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Step 1: Get initial and final estimates per order
window_spec = Window.partitionBy("order_id")

estimate_enriched = (
    estimate_silver
    .withColumn("min_version", F.min("version_no").over(window_spec))
    .withColumn("max_version", F.max("version_no").over(window_spec))
    .withColumn(
        "initial_estimate_amount",
        F.when(F.col("version_no") == F.col("min_version"), F.col("estimate_amount")).otherwise(None)
    )
    .withColumn(
        "final_estimate_amount",
        F.when(F.col("version_no") == F.col("max_version"), F.col("estimate_amount")).otherwise(None)
    )
    .groupBy("order_id")
    .agg(
        F.first("estimator_id", ignorenulls=True).alias("estimator_id"),
        F.max("initial_estimate_amount").alias("initial_estimate_amount"),
        F.max("final_estimate_amount").alias("final_estimate_amount")
    )
)

# Step 2: Join estimates to orders (keep all order columns)
fact_order_cycle = (
    order_silver
    .join(estimate_enriched, "order_id", "left")
    .drop("technician_name", "vehicle_make", "vehicle_model", "customer_name", "customer_phone")
)

print(f"fact_order_cycle: {fact_order_cycle.count()} rows")
display(fact_order_cycle.limit(5))

fact_order_cycle.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"fact_order_cycle")

In [0]:
invoice_df = invoice_silver.alias("i")
order_df = order_silver.alias("o")

fact_invoice = (
    invoice_df
    .join(order_df, "order_id", "left")
    .select(
        "i.invoice_id",
        "i.order_id",
        "o.store_id",
        "o.technician_id",
        "invoice_amount",
        "payment_mode",
        "currency",
        "invoice_date",
        "invoice_ts"
    )
)

print(f"fact_invoice: {fact_invoice.count()} rows")

fact_invoice.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"fact_invoice")

In [0]:
survey_df = customer_survey_silver.alias("s")
order_df = order_silver.alias("o")

fact_survey = (
    survey_df
    .join(order_df, "order_id", "left")
    .select(
        "s.survey_id",
        "s.order_id",
        "o.store_id",
        "o.technician_id",
        "responded_flag",
        "survey_sent_date",
        "survey_response_date",
        "overall_satisfaction_rating",
        "cleanliness_rating",
        "communication_rating",
        "work_quality_rating",
        "delivered_on_time_rating",
        F.datediff("survey_response_date", "survey_sent_date").alias("days_to_respond")
    )
)

print(f"fact_survey: {fact_survey.count()} rows")

fact_survey.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"fact_survey")